# SIMPL: Hand reaching demo

This demo applies SIMPL to somatosensory cortex recordings from a monkey performing a centre-out reaching task ([Chowdhury et al., 2020](https://doi.org/10.1016/j.neuron.2020.08.007)).

Motor cortex neurons encode both hand position and hand velocity — but which is the better latent space? We compare three choices of behavioural initialisation:

- **Part 1: Hand position (2D)** — initialise with (x, y) hand position
- **Part 2: Hand velocity (2D)** — initialise with (vx, vy) hand velocity
- **Part 3: Combined (4D)** — initialise with (x, y, vx, vy)

The raw data (NWB dataset `sub-Han`, 65 neurons) was preprocessed as follows:
1. Resampled to dt = 50 ms
2. Only active (non-bump) reaching conditions retained
3. Position and velocity normalised to [0, 1]

In [ ]:
try:
    from simpl import SIMPL, load_demo_data

    %load_ext autoreload
    %autoreload 2
except ImportError:
    %pip install git+https://github.com/TomGeorge1234/simpl.git
    from simpl import SIMPL, load_demo_data

import numpy as np

---
# Part 1: Hand position (2D)

The behavioural initialisation is 2D hand position (x, y). SIMPL will try to refine the decoded trajectory and learn position-tuned receptive fields.

## Load data

In [ ]:
data = load_demo_data("somatosensory_chowdhury2020.npz")

Y_all = data["Y"]
Xb_pos_all = data["Xb"]
Xb_vel_all = data["Xb_vel"]
time_all = data["time"]

# Trial timing (all 826 trials from the NWB file)
trial_start = data["trial_start"]  # index into time_all
trial_stop = data["trial_stop"]  # index into time_all
trial_move_onset = data["trial_move_onset"]  # index into time_all (-1 if no onset)
trial_cond_dir = data["trial_cond_dir"]  # reaching direction in degrees (8 dirs)
trial_result = data["trial_result"]  # 'R'each (success), 'A'bort, 'I'ncomplete
trial_is_bump = data["trial_is_bump"]  # True for passive bump trials

print(f"{'Y, spikes:': <35}{Y_all.shape}")
print(f"{'Xb, hand position:': <35}{Xb_pos_all.shape}")
print(f"{'Xb_vel, hand velocity:': <35}{Xb_vel_all.shape}")
print(f"{'time, time stamps:': <35}{time_all.shape}")
print(f"{'Trials (total):': <35}{len(trial_start)}")

# Hold out the last 60 seconds for prediction
dt = float(time_all[1] - time_all[0])
N_test = int(60 / dt)
N_train = len(time_all) - N_test

Y = Y_all[:N_train]
Xb_pos = Xb_pos_all[:N_train]
Xb_vel = Xb_vel_all[:N_train]
time = time_all[:N_train]

Y_test = Y_all[N_train:]
Xb_pos_test = Xb_pos_all[N_train:]
Xb_vel_test = Xb_vel_all[N_train:]

print(f"\nTrain/test split: {N_train} train ({N_train * dt:.0f}s), {N_test} test ({N_test * dt:.0f}s)")

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.cm import hsv


def overlay_reaches(axes, time_range=None):
    """Add faint coloured vspans showing reaching movements on trajectory axes.
    """
    if time_range is None:
        t0 = float(time_all[0])
        time_range = (t0, t0 + 120)

    for s, e, onset, cdir, result, bump in zip(
        trial_start, trial_stop, trial_move_onset, trial_cond_dir, trial_result, trial_is_bump
    ):
        if bump or onset < 0:
            continue
        t_onset = time_all[onset]
        t_stop = time_all[min(e, len(time_all) - 1)]
        if t_stop < time_range[0] or t_onset > time_range[1]:
            continue
        color = hsv(cdir / 360) if cdir >= 0 else "0.5"
        for ax in np.atleast_1d(axes):
            ax.axvspan(t_onset, t_stop, alpha=0.15, color=color, linewidth=0)

## Create and fit the SIMPL model

In [ ]:
pos_model = SIMPL(
    kernel_bandwidth=0.1,
    speed_prior=1,
    bin_size=0.02,
    env_pad=0.0,
)

In [ ]:
pos_model.fit(Y, Xb_pos, time, n_iterations=10)
pos_model.plot_fitting_summary();

## Results

In [ ]:
pos_model.results_

In [ ]:
axes = pos_model.plot_spikes(sort_by_spatial_information=True)
overlay_reaches(axes)

## Quick plots

In [ ]:
axes = pos_model.plot_latent_trajectory()
overlay_reaches(axes)

In [ ]:
pos_model.plot_receptive_fields(sort_by_spatial_information=True);

In [ ]:
pos_model.plot_all_metrics(show_neurons=False);

**Prediction** — decode held-out spikes using the fitted position-tuned receptive fields.

In [ ]:
pos_model.predict(Y_test)
pos_model.plot_prediction(Xb=Xb_pos_test);

---
# Part 2: Hand velocity (2D)

Now we initialise with 2D hand velocity (vx, vy) instead of position. The same spikes are used — only the behavioural initialisation changes. This tests whether motor cortex neurons are better described by velocity tuning.

## Create and fit the SIMPL model

In [ ]:
vel_model = SIMPL(
    kernel_bandwidth=0.1,
    speed_prior=1,
    bin_size=0.02,
    env_pad=0.0,
)

In [ ]:
vel_model.fit(Y, Xb_vel, time, n_iterations=10)
vel_model.plot_fitting_summary();

## Quick plots

In [ ]:
axes = vel_model.plot_latent_trajectory()
overlay_reaches(axes)

In [ ]:
vel_model.plot_receptive_fields(sort_by_spatial_information=True);

**Prediction** — decode held-out spikes using the fitted velocity-tuned receptive fields.

In [ ]:
vel_model.predict(Y_test)
vel_model.plot_prediction(Xb=Xb_vel_test);

---
# Part 3: Combined position + velocity (4D)

Finally, we initialise with all four variables (x, y, vx, vy) simultaneously. This lets SIMPL discover a joint position-velocity representation. Note the coarser `bin_size` — a 4D grid with fine bins would be prohibitively large.

## Load 4D behavioural initialisation

In [ ]:
Xb_4d = np.concatenate([Xb_pos, Xb_vel], axis=1)
Xb_4d_test = np.concatenate([Xb_pos_test, Xb_vel_test], axis=1)

print(f"{'Xb_4d, combined init:': <35}{Xb_4d.shape}")

## Create and fit the SIMPL model

In [ ]:
combined_model = SIMPL(
    kernel_bandwidth=0.09,
    speed_prior=1.5,
    bin_size=0.1,
    env_pad=0.0,
)

In [ ]:
combined_model.fit(Y, Xb_4d, time, n_iterations=10)
combined_model.plot_fitting_summary();

## Results

In [ ]:
combined_model.results_

In [ ]:
combined_model.plot_spikes(sort_by_spatial_information=True);

## Quick plots

In [ ]:
axes = combined_model.plot_latent_trajectory()
overlay_reaches(axes)

**Prediction** — decode held-out spikes using the fitted 4D receptive fields.

In [ ]:
combined_model.predict(Y_test)
combined_model.plot_prediction(Xb=Xb_4d_test);

---
# Compare models

Compare the validation log-likelihood across the three models to see which latent space best explains the neural data.

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(5, 3))
for model, label in [(pos_model, "Position (2D)"), (vel_model, "Velocity (2D)"), (combined_model, "Combined (4D)")]:
    r = model.results_
    iterations = r.iteration.values
    ax.plot(iterations, r.bits_per_spike_val.values, "o-", label=label, markersize=4)
ax.set_xlabel("Iteration")
ax.set_ylabel("Validation log P(Y|F)")
ax.legend()
fig.tight_layout()